In [ ]:
%%configure -f
{
  "vCores": 8,
  "defaultLakehouse": {
    "name": "<YOUR_LAKEHOUSE_NAME>",
    "id": "<YOUR_LAKEHOUSE_ID>",
    "workspaceId": "<YOUR_WORKSPACE_ID>"
  }
}

# fabric-rlm: an API tour

A practical walk through the `fabric_rlm` API: how to instantiate an `RLM`,
the three ways to declare a task, the constructor parameters worth knowing,
how to run it, and how to get, save, and inspect traces from a Fabric Lakehouse.

The running example is a task a single model call gets wrong but an `RLM`
solves exactly, even with a small model, because it writes and runs Python
instead of guessing.

`%%configure` must stay the first cell. Replace the lakehouse placeholders with
your own, or drop the `defaultLakehouse` block to run without one.

## Setup

In [ ]:
%pip install -q fabric-rlm

In [ ]:
import json, random
from pathlib import Path

import fabric_rlm
from fabric_rlm import RLM, FabricLM, Trajectory

print("fabric_rlm", fabric_rlm.__version__)

# Use a small model on purpose. Name a deployment your workspace actually
# exposes (see the Fabric prebuilt-models list); gpt-4.1-mini is a safe default.
MODEL = "gpt-4.1-mini"
lm = FabricLM(MODEL, temperature=0.0)

## What a single model call gets wrong

180 daily cash movements, starting balance 0. Three questions: the final
balance, how many days the running balance was negative, and the lowest the
balance ever fell to. All three require carrying an exact running total across
180 rows. A single call has to do this in its head.

We compute the ground truth here so we can check both approaches against it.

In [ ]:
random.seed(7)
amounts, running, balance = [], [], 0
for _ in range(180):
    amt = random.randint(-450, 400)
    balance += amt
    amounts.append(amt)
    running.append(balance)

truth = {
    "final_balance": running[-1],
    "negative_days": sum(1 for b in running if b < 0),
    "min_balance": min(running),
}
print("ground truth:", truth)

In [ ]:
lines = "\n".join(f"day {i+1}: {a:+d}" for i, a in enumerate(amounts))
prompt = (
    "These are 180 daily cash movements; the starting balance is 0.\n"
    f"{lines}\n\n"
    "Reply with JSON only: "
    '{"final_balance": int, "negative_days": int, "min_balance": int} '
    "where negative_days is the number of days the running balance was below 0 "
    "and min_balance is the lowest the running balance ever reached."
)
raw = lm(messages=[{"role": "user", "content": prompt}])
direct = raw[0] if isinstance(raw, list) else raw
print(direct)

The model approximates. Each answer requires carrying an exact running total
across 180 rows, which is where a single pass drifts.

## The same task with an RLM

`RLM` gives the model a Python sandbox and a turn loop: it writes code, sees
the output, and calls `SUBMIT(...)` when done. The simplest way to declare a
task is a string signature, `"inputs -> outputs"` (the dspy convention).

In [ ]:
rlm = RLM("amounts -> final_balance, negative_days, min_balance", lm=lm, max_turns=6)
result = rlm.run({"amounts": amounts})

print("submitted:", result.submitted, "| turns:", result.n_turns)
print("answer   :", result.outputs)
print("correct  :", result.outputs == truth)

Exact, in a few turns, with the same small model. It wrote a loop instead of
guessing.

## Three ways to declare a task

The `signature` argument accepts a string or a dspy `Signature` class, and
`RLM.task(...)` covers free-form tasks. All three work with the default
engine.

In [ ]:
# 1) String signature: inputs and outputs, separated by ->.
rlm = RLM("text -> word_count, unique_words", lm=lm, max_turns=4)
print(rlm.run({"text": "the cat sat on the mat the cat ran"}).outputs)

In [ ]:
# 2) A dspy.Signature class. Its docstring becomes the task description;
#    the InputField / OutputField names define the namespace and the required
#    output fields.
import dspy


class CountWords(dspy.Signature):
    """Count the total and the distinct words in the text."""

    text = dspy.InputField()
    word_count = dspy.OutputField(desc="total number of words")
    unique_words = dspy.OutputField(desc="number of distinct words")


rlm = RLM(CountWords, lm=lm, max_turns=4)
print(rlm.run({"text": "the cat sat on the mat the cat ran"}).outputs)

If you would rather not depend on dspy, the same described task is a plain
string signature or `RLM.task`. Per-field guidance that the Signature carried
in `desc=...` simply goes into the task text instead.

In [ ]:
# CountWords without dspy: no import, no class. The output names come from
# `outputs`; the guidance goes in `task`.
rlm = RLM.task(
    task="Count the total number of words and the number of distinct words.",
    inputs={"text": "the cat sat on the mat the cat ran"},
    outputs=["word_count", "unique_words"],
    lm=lm,
    max_turns=4,
)
print(rlm.run().outputs)

In [ ]:
# 3) RLM.task: a free-form task, explicit inputs, and declared output fields.
#    Inputs passed here are bound for you, so run() needs no arguments.
rlm = RLM.task(
    task="Return the median and the 90th percentile of the values.",
    inputs={"values": [4, 8, 15, 16, 23, 42, 1, 99, 7, 3]},
    outputs=["median", "p90"],
    lm=lm,
    max_turns=4,
)
print(rlm.run().outputs)

## Constructor parameters worth knowing

| Parameter | What it does |
|---|---|
| `lm` | The driving model. A string, a dict spec, a dspy LM, or any callable. |
| `sub_lm` | Model the sandbox uses for nested `predict(...)` calls. |
| `max_turns` | Hard cap on code/feedback rounds before the loop stops. |
| `timeout` | Per-worker execution timeout, in seconds. |
| `skills` | Named playbooks to preload (e.g. `["pdf_document_analysis"]`). |
| `enable_skill_autoloading` | Let the model pull in skills mid-run. |
| `enable_verifier` | Run skill verifiers on the submitted payload. |
| `output_validator` | Your own check on the payload; raise `AssertionError` to force a repair turn. |
| `stuck_loop_threshold` | Abort after N identical failing turns (default 3). |
| `verbose` | Print each turn's code and output as it runs. |
| `engine` | Loop implementation; defaults to `auto` (`default` unless tools are supplied). |

`output_validator` is the most useful one to know: it turns a domain rule into
an automatic repair.

In [ ]:
def check(payload):
    assert payload["p90"] >= payload["median"], "p90 must be >= median"


rlm = RLM.task(
    task="Return the median and the 90th percentile of the values.",
    inputs={"values": [4, 8, 15, 16, 23, 42, 1, 99, 7, 3]},
    outputs=["median", "p90"],
    lm=lm,
    max_turns=6,
    timeout=120,
    output_validator=check,
)
r = rlm.run()
print(r.outputs, "| turns:", r.n_turns)

## Running: run() vs calling the instance

`run(inputs)` takes an explicit dict. Calling the instance with keyword
arguments is shorthand for the same thing.

In [ ]:
rlm = RLM("a, b -> total", lm=lm, max_turns=3)

print(rlm.run({"a": 21, "b": 21}).outputs)   # explicit dict
print(rlm(a=21, b=21).outputs)                # keyword shorthand

## Traces: the trajectory

Every run returns an `RLMResult`. Beyond `submitted` and `outputs`, it carries
a full `trajectory`: one `TurnRecord` per turn with the code the model ran, its
stdout/stderr, any error, the submit payload, and per-turn token usage.

In [ ]:
result = rlm.run({"a": 120, "b": 45})

print("turns:", result.n_turns, "| submitted:", result.submitted)
for t in result.trajectory.turns:
    first_line = (t.stdout.strip().splitlines() or [""])[0]
    print(f"  turn {t.turn}: submitted={t.submitted} "
          f"code={len(t.code)}c stdout={first_line!r} tokens={t.total_tokens}")

print("\n--- code the model ran on the final turn ---")
print(result.trajectory.turns[-1].code)

## Save traces to a Lakehouse

A trajectory serializes to JSONL (one metadata line, then one line per turn)
and to Markdown. A mounted Lakehouse `Files` path is just a file path.

In [ ]:
out_dir = Path("/lakehouse/default/Files/rlm_traces")
out_dir.mkdir(parents=True, exist_ok=True)

jsonl_path = out_dir / "balance.jsonl"
result.trajectory.write_jsonl(jsonl_path)
result.trajectory.write_markdown(out_dir / "balance.md")
print("wrote", jsonl_path)

## Reload and inspect a saved trace

`Trajectory.from_jsonl` reads back a local path, an `abfss://` URI (via
notebookutils, no extra dependency), a file-like object, or a list of dicts
from a Spark DataFrame.

In [ ]:
loaded = Trajectory.from_jsonl(jsonl_path)
print("turns:", len(loaded.turns), "| metadata:", loaded.metadata)

# From OneLake directly:
# loaded = Trajectory.from_jsonl(
#     "abfss://<workspace>@onelake.dfs.fabric.microsoft.com/"
#     "<lakehouse>.Lakehouse/Files/rlm_traces/balance.jsonl")

# Or pretty-print from a terminal:
#   python -m fabric_rlm.replay path/to/balance.jsonl

## Replay a trace offline, with no API calls

`replay_trajectory` drives the real loop from a recording using the stored
responses and worker results: no model, no subprocess. A recorded trajectory
becomes a deterministic regression test, and a behavior change surfaces as a
`DivergenceError` rather than a silent difference.

In [ ]:
from fabric_rlm import replay_trajectory

offline = RLM("a, b -> total", lm=lm, max_turns=3)
replayed = replay_trajectory(offline, loaded)
print("replayed:", replayed.outputs, "| API calls: 0")

## Cost and timing

The result aggregates token usage and timing across all turns. Fields are
`None` when the backend did not report them.

In [ ]:
print("prompt tokens    :", result.total_prompt_tokens)
print("completion tokens:", result.total_completion_tokens)
print("LM seconds       :", result.total_lm_seconds)
print("worker seconds   :", result.total_worker_seconds)

## Where to go next

- Long-document and PDF workloads: the `pdf_document_analysis` skill and the
  notebooks in `examples/notebooks/`.
- Golden-trajectory testing: `examples/replay_golden_trajectories.py`.
- Validators and verifiers for enforcing output contracts.